# Pipeline v2 Workspace

Explore the ResultsStore, inspect completed runs, and load result blobs.

In [1]:
%reload_ext autoreload
%autoreload 2
from dimensionality_manuscript.registry import RegistryPaths
from dimensionality_manuscript_v2 import ResultsStore, CVPCAConfig, get_data_config
from dimensionality_manuscript_v2.scripts.status import status

## Store overview

In [3]:
store = ResultsStore(RegistryPaths().pipeline_v2_db_path)
status()

Store: D:\localData\dimensionality-manuscript\cache\pipeline_v2\results.db
Total rows: 7485
Snapshots: 4

                              count  stored  sessions  data_configs                          earliest                            latest
analysis_type schema_version                                                                                                           
cvpca         v1               4654    4654       148             1  2026-03-09T15:03:22.216575+00:00  2026-03-10T15:31:39.024107+00:00
population    v1                149     149       149             1  2026-03-10T15:14:19.696067+00:00  2026-03-10T15:30:42.303680+00:00
regression    v1               1490    1490       149             1  2026-03-09T15:06:16.237990+00:00  2026-03-10T15:29:04.864417+00:00
subspace      v1                607     607       149             1  2026-03-10T15:14:18.554047+00:00  2026-03-10T15:30:48.210735+00:00
svca          v1                585     585       149             1  2026-03-1

## Browse the summary table

In [3]:
df = store.summary_table(as_dataframe=True)
df

,result_uid,session_id,data_key,analysis_key,data_summary,analysis_summary,analysis_type,schema_version,result_stored,snapshot_path,computed_at
0,a0a9891c2857a347,ATL012.2023-01-20.702,ce76ae68bcb57797,a5f952466d9772b3,default,cvpca_center=True_norm=True_fast=True_rel=None...,cvpca,v1,1,D:\localData\dimensionality-manuscript\cache\p...,2026-03-09T14:05:26.390461+00:00
1,483846b577c0a947,ATL012.2023-01-20.702,ce76ae68bcb57797,b8e0e4ea18454d91,default,cvpca_center=True_norm=True_fast=True_rel=0.2_...,cvpca,v1,1,D:\localData\dimensionality-manuscript\cache\p...,2026-03-09T14:05:32.026189+00:00
2,dd6dabec8a712fc2,ATL012.2023-01-20.702,ce76ae68bcb57797,77377c764601496d,default,cvpca_center=True_norm=True_fast=True_rel=0.2_...,cvpca,v1,1,D:\localData\dimensionality-manuscript\cache\p...,2026-03-09T14:05:35.228967+00:00
3,0e52d8dff775efda,ATL012.2023-01-20.702,ce76ae68bcb57797,41442170904f9abe,default,cvpca_center=True_norm=False_fast=True_rel=Non...,cvpca,v1,1,D:\localData\dimensionality-manuscript\cache\p...,2026-03-09T14:05:44.784327+00:00
4,3ae1c46b5b2e5454,ATL012.2023-01-20.702,ce76ae68bcb57797,24ee9d6e07f2f9b0,default,cvpca_center=True_norm=False_fast=True_rel=Non...,cvpca,v1,1,D:\localData\dimensionality-manuscript\cache\p...,2026-03-09T14:05:54.338077+00:00
5,9d5c8a2433d7fd02,ATL012.2023-01-20.702,ce76ae68bcb57797,301e91704c31cede,default,cvpca_center=True_norm=True_fast=False_rel=Non...,cvpca,v1,1,D:\localData\dimensionality-manuscript\cache\p...,2026-03-09T14:06:04.582831+00:00
6,f81e11930b24a353,ATL012.2023-01-20.702,ce76ae68bcb57797,d9df6b6c5202a91c,default,cvpca_center=True_norm=True_fast=False_rel=Non...,cvpca,v1,1,D:\localData\dimensionality-manuscript\cache\p...,2026-03-09T14:06:14.146424+00:00
7,8f815cb12bfc1c02,ATL012.2023-01-20.702,ce76ae68bcb57797,15c8aa1da526008e,default,cvpca_center=True_norm=True_fast=False_rel=0.2...,cvpca,v1,1,D:\localData\dimensionality-manuscript\cache\p...,2026-03-09T14:06:17.052515+00:00
8,4a32f61199f4ddf8,ATL012.2023-01-20.702,ce76ae68bcb57797,2ee7570fd1761f1c,default,cvpca_center=True_norm=True_fast=False_rel=0.2...,cvpca,v1,1,D:\localData\dimensionality-manuscript\cache\p...,2026-03-09T14:06:19.992397+00:00


## Load a result by reconstructing configs from the table

Pick a row, reconstruct the `DataConfig` and `CVPCAConfig` that produced it, and retrieve the result blob.

In [4]:
# Pick the first row as an example
row = df.iloc[0]
print(f"result_uid:       {row['result_uid']}")
print(f"session_id:       {row['session_id']}")
print(f"data_summary:     {row['data_summary']}")
print(f"analysis_summary: {row['analysis_summary']}")
print(f"computed_at:      {row['computed_at']}")

result_uid:       a0a9891c2857a347
session_id:       ATL012.2023-01-20.702
data_summary:     default
analysis_summary: cvpca_center=True_norm=True_fast=True_rel=None_frac=0.05_smooth=3.0_bins=100_v1
computed_at:      2026-03-09T14:05:26.390461+00:00


In [5]:
# Reconstruct the configs that produced this row
dcfg = get_data_config(row["data_summary"])
acfg = CVPCAConfig.from_key(row["analysis_key"])

print(f"DataConfig:     {dcfg}")
print(f"AnalysisConfig: {acfg}")

DataConfig:     DataConfig(name='default', speed_threshold=1.0, time_split_groups=4, time_split_relative_size=(4, 4, 1, 1), time_split_chunks_per_group=10, time_split_num_buffer=3, cell_split_force_even=False, spks_type='oasis')
AnalysisConfig: CVPCAConfig(schema_version='v1', center=True, normalize=True, use_fast_sampling=True, reliability_threshold=None, fraction_active_threshold=0.05, fixed_smooth_width=3.0, num_bins=100)


In [6]:
# Load the result blob using the reconstructed configs
result = store.get(row["session_id"], dcfg, acfg)
print(f"Result type: {type(result)}")
print(f"Result keys: {list(result.keys())}")

# Or load directly by uid (no config reconstruction needed)
result2 = store.get_by_uid(row["result_uid"])
assert result2.keys() == result.keys()

Result type: <class 'dict'>
Result keys: ['trial_folds', 'reg_covariances', 'reg_covariances_fixed', 'org_covariances', 'org_smooth_covariances', 'org_fixed_smooth_covariances', 'pca_covariances', 'pca_smooth_covariances', 'pca_fixed_smooth_covariances', 'saved_leg_covariances', 'smoothing_widths', 'fixed_smoothing_width']


## Inspect result contents

In [7]:
import numpy as np

for key, val in result.items():
    if isinstance(val, np.ndarray):
        print(f"  {key}: ndarray {val.shape} {val.dtype}")
    else:
        print(f"  {key}: {type(val).__name__} = {val}")

  trial_folds: list = [[5, 12, 20, 6, 11, 24, 7, 1, 17], [3, 27, 8, 2, 10, 18, 21, 0, 23], [26, 19, 16, 22, 13, 25, 14, 9, 15, 4]]
  reg_covariances: ndarray (100,) float32
  reg_covariances_fixed: ndarray (100,) float32
  org_covariances: ndarray (100,) float32
  org_smooth_covariances: ndarray (100,) float32
  org_fixed_smooth_covariances: ndarray (100,) float32
  pca_covariances: ndarray (100,) float32
  pca_smooth_covariances: ndarray (100,) float32
  pca_fixed_smooth_covariances: ndarray (100,) float32
  saved_leg_covariances: ndarray (80,) float64
  smoothing_widths: ndarray (4205,) float32
  fixed_smoothing_width: float = 3.0


## Compare with legacy workflow results

Load the same result from the old `measure_cvpca.py` file-based cache and compare side-by-side. The trial folds are randomized independently, so results won't match exactly — but the structure and scale should agree.

In [8]:
import joblib
from dimensionality_manuscript.registry import RegistryPaths


def get_legacy_filepath(session_id: str, acfg: CVPCAConfig):
    """Reconstruct the legacy measure_cvpca.py filepath from a CVPCAConfig."""
    from pathlib import Path

    rp = RegistryPaths()
    name = session_id
    if not acfg.center:
        name += "_notcentered"
    if acfg.use_fast_sampling:
        name += "_fast"
    if not acfg.normalize:
        name += "_nonorm"
    if acfg.reliability_threshold is not None:
        name += f"_rel{acfg.reliability_threshold:.2f}"
    if acfg.fraction_active_threshold is not None:
        name += f"_frac{acfg.fraction_active_threshold:.2f}"
    return rp.measure_cvpca_path / f"{name}.pkl"


# Pick a row to compare (change iloc index to pick a different one)
iloc = 0
row = df.iloc[iloc]
dcfg = get_data_config(row["data_summary"])
acfg = CVPCAConfig.from_key(row["analysis_key"])

# Load pipeline v2 result
v2_result = store.get(row["session_id"], dcfg, acfg)

# Load legacy result
legacy_path = get_legacy_filepath(row["session_id"], acfg)
print(f"Legacy path: {legacy_path}")
print(f"Exists: {legacy_path.exists()}")

if legacy_path.exists():
    legacy_result = joblib.load(legacy_path)
    print(f"\nv2 keys:     {sorted(v2_result.keys())}")
    print(f"legacy keys: {sorted(legacy_result.keys())}")
else:
    legacy_result = None
    print("Legacy result not found — run measure_cvpca.py first for this config.")

Legacy path: D:\localData\dimensionality-manuscript\cache\measure_cvpca\ATL012.2023-01-20.702_fast_frac0.05.pkl
Exists: True

v2 keys:     ['fixed_smoothing_width', 'org_covariances', 'org_fixed_smooth_covariances', 'org_smooth_covariances', 'pca_covariances', 'pca_fixed_smooth_covariances', 'pca_smooth_covariances', 'reg_covariances', 'reg_covariances_fixed', 'saved_leg_covariances', 'smoothing_widths', 'trial_folds']
legacy keys: ['fixed_smoothing_width', 'org_covariances', 'org_fixed_smooth_covariances', 'org_smooth_covariances', 'pca_covariances', 'pca_fixed_smooth_covariances', 'pca_smooth_covariances', 'reg_covariances', 'reg_covariances_fixed', 'saved_leg_covariances', 'smoothing_widths', 'trial_folds']


In [9]:
# Compare array shapes and print per-key stats
if legacy_result is not None:
    shared_keys = sorted(set(v2_result.keys()) & set(legacy_result.keys()))
    for key in shared_keys:
        v2_val = v2_result[key]
        leg_val = legacy_result[key]
        if isinstance(v2_val, np.ndarray) and isinstance(leg_val, np.ndarray):
            # Truncate to shared length (legacy may have fewer bins)
            n = min(len(v2_val), len(leg_val))
            corr = np.corrcoef(v2_val[:n].astype(float), leg_val[:n].astype(float))[0, 1]
            print(f"  {key:35s}  v2={v2_val.shape}  legacy={leg_val.shape}  corr={corr:.4f}")
        else:
            print(f"  {key:35s}  v2={type(v2_val).__name__}  legacy={type(leg_val).__name__}")

  fixed_smoothing_width                v2=float  legacy=int
  org_covariances                      v2=(100,)  legacy=(100,)  corr=0.9957
  org_fixed_smooth_covariances         v2=(100,)  legacy=(100,)  corr=0.9977
  org_smooth_covariances               v2=(100,)  legacy=(100,)  corr=0.9996
  pca_covariances                      v2=(100,)  legacy=(100,)  corr=0.9996
  pca_fixed_smooth_covariances         v2=(100,)  legacy=(100,)  corr=0.9999
  pca_smooth_covariances               v2=(100,)  legacy=(100,)  corr=1.0000
  reg_covariances                      v2=(100,)  legacy=(100,)  corr=0.9989
  reg_covariances_fixed                v2=(100,)  legacy=(100,)  corr=0.9971
  saved_leg_covariances                v2=(80,)  legacy=(80,)  corr=1.0000
  smoothing_widths                     v2=(4205,)  legacy=(4205,)  corr=0.6611
  trial_folds                          v2=list  legacy=list
